# DCGAN — Handwritten Digit Generation (MNIST)
**Reference:** Radford et al., *Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks*, arXiv:1511.06434

---
### Quick-start on Google Colab
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells (**Runtime → Run all**)
3. Training takes ~15–30 min on Colab T4


## 1. Setup & Imports

In [ ]:
import os, time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

os.makedirs('samples', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

## 2. Hyper-parameters

In [ ]:
LATENT_DIM   = 100     # Size of noise vector z
GEN_FEATURES = 64      # Generator base feature maps
DIS_FEATURES = 64      # Discriminator base feature maps
IMAGE_SIZE   = 64      # Resize MNIST 28×28 → 64×64
CHANNELS     = 1       # Grayscale
BATCH_SIZE   = 128
LR           = 0.0002
BETA1        = 0.5     # Adam β1 (paper recommendation)
BETA2        = 0.999
NUM_EPOCHS   = 25

## 3. Dataset — MNIST

In [ ]:
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  # → [-1, 1]
])

dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print(f'Dataset size: {len(dataset):,} images')

# Visualise some real samples
real_batch, _ = next(iter(dataloader))
plt.figure(figsize=(8, 8))
plt.title('Real MNIST Samples')
plt.imshow(
    np.transpose(vutils.make_grid(real_batch[:64], nrow=8, normalize=True).numpy(), (1, 2, 0)),
    cmap='gray'
)
plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Weight Initialisation (DCGAN §4)
All Conv and BatchNorm weights initialised from N(0, 0.02).

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

## 5. Generator Architecture

**z** (100-dim noise) → 4 transposed convolutions → **64×64 grayscale image**

| Layer | In → Out | Kernel | Stride | Padding | Activation |
|-------|----------|--------|--------|---------|------------|
| 1 | z → 512ch | 4×4 | 1 | 0 | BN + ReLU |
| 2 | 512 → 256ch | 4×4 | 2 | 1 | BN + ReLU |
| 3 | 256 → 128ch | 4×4 | 2 | 1 | BN + ReLU |
| 4 | 128 → 64ch | 4×4 | 2 | 1 | BN + ReLU |
| out | 64 → 1ch | 4×4 | 2 | 1 | **Tanh** |

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        nz, ngf, nc = LATENT_DIM, GEN_FEATURES, CHANNELS
        self.net = nn.Sequential(
            # z: (nz) × 1 × 1
            nn.ConvTranspose2d(nz,    ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),             # → 4×4

            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),             # → 8×8

            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),             # → 16×16

            nn.ConvTranspose2d(ngf*2, ngf,   4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),   nn.ReLU(True),             # → 32×32

            nn.ConvTranspose2d(ngf,   nc,    4, 2, 1, bias=False),
            nn.Tanh()                                         # → 64×64
        )

    def forward(self, z):
        return self.net(z)

G = Generator().to(device)
G.apply(weights_init)
print(G)
print(f'Generator parameters: {sum(p.numel() for p in G.parameters()):,}')

## 6. Discriminator Architecture

**64×64 image** → 4 strided convolutions → **scalar probability**

| Layer | In → Out | Kernel | Stride | Activation |
|-------|----------|--------|--------|------------|
| 1 | 1 → 64ch | 4×4 | 2 | LeakyReLU(0.2) |
| 2 | 64 → 128ch | 4×4 | 2 | BN + LeakyReLU |
| 3 | 128 → 256ch | 4×4 | 2 | BN + LeakyReLU |
| 4 | 256 → 512ch | 4×4 | 2 | BN + LeakyReLU |
| out | 512 → 1 | 4×4 | 1 | **Sigmoid** |

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        ndf, nc = DIS_FEATURES, CHANNELS
        self.net = nn.Sequential(
            # No BatchNorm on first layer (per DCGAN paper)
            nn.Conv2d(nc,    ndf,   4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),                 # → 32×32

            nn.Conv2d(ndf,   ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),   # → 16×16

            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),   # → 8×8

            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, inplace=True),   # → 4×4

            nn.Conv2d(ndf*8, 1,    4, 1, 0, bias=False),
            nn.Sigmoid()                                     # → scalar
        )

    def forward(self, x):
        return self.net(x).view(-1)

D = Discriminator().to(device)
D.apply(weights_init)
print(D)
print(f'Discriminator parameters: {sum(p.numel() for p in D.parameters()):,}')

## 7. Loss Function & Optimisers

In [ ]:
criterion = nn.BCELoss()

optim_G = optim.Adam(G.parameters(), lr=LR, betas=(BETA1, BETA2))
optim_D = optim.Adam(D.parameters(), lr=LR, betas=(BETA1, BETA2))

# Fixed noise: always visualise the same latent codes → shows G improving over time
fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=device)

print('Optimisers ready.')

## 8. Training Loop

**Minimax game:**
```
min_G  max_D  E[log D(x)] + E[log(1 − D(G(z)))]
```
In practice we train G to **maximise** `log D(G(z))` (non-saturating form).

In [ ]:
history = {'G_loss': [], 'D_loss': [], 'D_real': [], 'D_fake': []}
img_list = []  # store grids for animation

print('Starting training …\n')
for epoch in range(1, NUM_EPOCHS + 1):
    g_loss_sum = d_loss_sum = dr_sum = df_sum = 0.0
    t0 = time.time()

    for real_imgs, _ in dataloader:
        real_imgs = real_imgs.to(device)
        b = real_imgs.size(0)
        ones  = torch.ones(b,  device=device)
        zeros = torch.zeros(b, device=device)

        # ── Train Discriminator ──────────────
        D.zero_grad()
        # Real
        loss_D_real = criterion(D(real_imgs), ones)
        loss_D_real.backward()
        dr_sum += D(real_imgs).mean().item()
        # Fake
        noise = torch.randn(b, LATENT_DIM, 1, 1, device=device)
        fake  = G(noise)
        loss_D_fake = criterion(D(fake.detach()), zeros)
        loss_D_fake.backward()
        df_sum += D(fake.detach()).mean().item()
        optim_D.step()
        d_loss_sum += (loss_D_real + loss_D_fake).item()

        # ── Train Generator ──────────────────
        G.zero_grad()
        loss_G = criterion(D(fake), ones)  # want D to say 'real'
        loss_G.backward()
        optim_G.step()
        g_loss_sum += loss_G.item()

    n = len(dataloader)
    history['G_loss'].append(g_loss_sum / n)
    history['D_loss'].append(d_loss_sum / n)
    history['D_real'].append(dr_sum / n)
    history['D_fake'].append(df_sum / n)

    elapsed = time.time() - t0
    print(f'Epoch [{epoch:3d}/{NUM_EPOCHS}]  '
          f'D: {d_loss_sum/n:.4f}  G: {g_loss_sum/n:.4f}  '
          f'D(x): {dr_sum/n:.3f}  D(G(z)): {df_sum/n:.3f}  '
          f'[{elapsed:.1f}s]')

    # Save sample grid every 5 epochs
    if epoch % 5 == 0 or epoch == NUM_EPOCHS:
        G.eval()
        with torch.no_grad():
            fake_fixed = G(fixed_noise).cpu()
        G.train()
        img_list.append(fake_fixed)
        vutils.save_image(fake_fixed, f'samples/epoch_{epoch:03d}.png', nrow=8, normalize=True)

print('\nTraining complete!')

## 9. Training Loss Curves

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history['D_loss'], label='Discriminator', color='#e74c3c')
axes[0].plot(epochs, history['G_loss'], label='Generator',     color='#3498db')
axes[0].set_title('GAN Training Losses')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['D_real'], label='D(real)',  color='#27ae60')
axes[1].plot(epochs, history['D_fake'], label='D(G(z))', color='#e67e22')
axes[1].axhline(0.5, linestyle='--', color='gray', alpha=0.5, label='Equilibrium')
axes[1].set_title('Discriminator Confidence')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Average D output')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('samples/loss_curve.png', dpi=150)
plt.show()

## 10. Generated Samples — Final Epoch

In [ ]:
G.eval()
with torch.no_grad():
    noise = torch.randn(64, LATENT_DIM, 1, 1, device=device)
    fake_imgs = G(noise).cpu()

plt.figure(figsize=(10, 10))
plt.title('Generated Handwritten Digits (final epoch)', fontsize=14)
plt.imshow(
    np.transpose(vutils.make_grid(fake_imgs, nrow=8, normalize=True).numpy(), (1, 2, 0)),
    cmap='gray'
)
plt.axis('off')
plt.tight_layout()
plt.savefig('samples/final_generated.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to samples/final_generated.png')

## 11. Side-by-Side: Real vs Generated

In [ ]:
real_batch, _ = next(iter(dataloader))
real_batch = real_batch[:32]

G.eval()
with torch.no_grad():
    noise = torch.randn(32, LATENT_DIM, 1, 1, device=device)
    fake_batch = G(noise).cpu()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, imgs, title in zip(axes,
                            [real_batch, fake_batch],
                            ['Real MNIST', 'DCGAN Generated']):
    grid = np.transpose(vutils.make_grid(imgs, nrow=8, normalize=True).numpy(), (1, 2, 0))
    ax.imshow(grid, cmap='gray')
    ax.set_title(title, fontsize=14)
    ax.axis('off')

plt.tight_layout()
plt.savefig('samples/real_vs_generated.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Save Checkpoint

In [ ]:
torch.save({
    'epoch': NUM_EPOCHS,
    'G_state': G.state_dict(),
    'D_state': D.state_dict(),
    'optim_G': optim_G.state_dict(),
    'optim_D': optim_D.state_dict(),
    'history': history,
}, f'checkpoints/dcgan_mnist_epoch{NUM_EPOCHS}.pt')
print('Checkpoint saved!')